# MultiMend — Kaggle A→Z baseline reproduction (first benchmark: QuixBugs-Python)

**Scope:** official MultiMend repository + official fine-tuned checkpoints + official QuixBugs benchmark.

This notebook deliberately does **not** run all six benchmarks. It first validates the complete pipeline on QuixBugs-Python:

`buggy/correct programs → official bug-line/context extraction → local RAG → CodeT5 checkpoint ensemble → 100 candidates/checkpoint → combine/deduplicate → patch validation → CSV/JSON summary`

No OpenAI/Claude/Gemini API is used. The model and the RAG embedding model are downloaded locally from Hugging Face.


In [7]:
from pathlib import Path
import shutil

repo = Path("/kaggle/working/MultiMend28")

if repo.exists():
    shutil.rmtree(repo)

print("Old repository removed.")

Old repository removed.


In [8]:
# Cell 1 — Kaggle setup
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ThaoTo2806/MultiMend28.git"
REPO = Path("/kaggle/working/MultiMend28")

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("Repository:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Python:", sys.version)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("Torch will be checked after dependency installation.")


Cloning into '/kaggle/working/MultiMend28'...


Repository: /kaggle/working/MultiMend28
Commit: 08c0c75f5331f54616a083267e501f21713c6047
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.4.0+cu121
CUDA available: True
GPU: Tesla T4


In [5]:
# Cell 1 — Kaggle environment
import os, sys, subprocess, textwrap, json, pathlib, shutil, time
from pathlib import Path

print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import error:", repr(e))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
CUDA_VISIBLE_DEVICES: None
Torch: 2.4.0+cu121
CUDA available: True
GPU: Tesla T4


In [9]:
# Cell 2 — Install the repository dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "-r", "requirements.txt", "more-itertools", "pygments",
    ],
    check=True,
)
print("Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 1.1 MB/s eta 0:00:00
Dependencies installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.27.1 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.4.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.27.1 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.5 which is incompatible.


In [10]:
# Download QuixBugs if it is not already present
QB_DIR = REPO / "benchmarks" / "QuixBugs"

if not (QB_DIR / "python_programs").is_dir():
    QB_DIR.parent.mkdir(parents=True, exist_ok=True)

    !git clone https://github.com/jkoppel/QuixBugs.git "{QB_DIR}"

print("QuixBugs:", QB_DIR)
print("Python programs:", len(list((QB_DIR / "python_programs").glob("*.py"))))


Cloning into '/kaggle/working/MultiMend28/benchmarks/QuixBugs'...
remote: Enumerating objects: 1123, done.
remote: Counting objects: 100% (478/478), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 1123 (delta 340), reused 300 (delta 300), pack-reused 645 (from 1)
Receiving objects: 100% (1123/1123), 1.14 MiB | 20.80 MiB/s, done.
Resolving deltas: 100% (648/648), done.
QuixBugs: /kaggle/working/MultiMend28/benchmarks/QuixBugs
Python programs: 50


In [11]:
# Cell 3 — Prepare benchmark data and checkpoints

import os
import json
import shutil
from pathlib import Path

# ================================================================
# 1. Repository
# ================================================================

REPO = Path("/kaggle/working/MultiMend28")
REPO.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)

print("Repository:", REPO)


# ================================================================
# 2. Prepare QuixBugs benchmark
# ================================================================

QB_DIR = REPO / "benchmarks" / "QuixBugs"

# Download QuixBugs only if it is missing.
if not (QB_DIR / "python_programs").is_dir():
    print("\nQuixBugs not found. Downloading...")

    QB_DIR.parent.mkdir(parents=True, exist_ok=True)

    result = os.system(
        f'git clone https://github.com/jkoppel/QuixBugs.git "{QB_DIR}"'
    )

    if result != 0:
        raise RuntimeError(
            "Failed to download QuixBugs. "
            "Please enable Internet in Kaggle or attach QuixBugs as a dataset."
        )

# Verify benchmark structure.
if not (QB_DIR / "python_programs").is_dir():
    raise FileNotFoundError(
        f"QuixBugs python_programs directory is missing: {QB_DIR}"
    )

if not (QB_DIR / "correct_python_programs").is_dir():
    raise FileNotFoundError(
        f"QuixBugs correct_python_programs directory is missing: {QB_DIR}"
    )

buggy_programs = sorted(
    (QB_DIR / "python_programs").glob("*.py")
)

correct_programs = sorted(
    (QB_DIR / "correct_python_programs").glob("*.py")
)

print("\nQuixBugs benchmark:", QB_DIR)
print("Buggy programs:", len(buggy_programs))
print("Correct programs:", len(correct_programs))


# ================================================================
# 3. Locate MultiMend checkpoints
# ================================================================

MODEL_DIR = REPO / "models" / "multimend-codet5-small"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Optional manual override.
# Example:
# os.environ["MULTIMEND_CHECKPOINT_SOURCE"] = "/kaggle/input/..."
configured_source = os.environ.get("MULTIMEND_CHECKPOINT_SOURCE")


if configured_source:
    print("\nUsing configured checkpoint source:")
    print(configured_source)

    source = Path(configured_source)

    checkpoint_dirs = sorted(
        p for p in source.rglob("checkpoint-*")
        if p.is_dir() and (p / "trainer_state.json").is_file()
    )

else:
    # Automatically search all Kaggle input datasets.
    print("\nSearching /kaggle/input/datasets for checkpoints...")

    checkpoint_dirs = sorted(
        p.parent
        for p in Path("/kaggle/input/datasets").rglob("trainer_state.json")
        if p.parent.name.startswith("checkpoint-")
    )


# Remove duplicates while preserving order.
checkpoint_dirs = sorted(set(checkpoint_dirs))


if not checkpoint_dirs:
    raise FileNotFoundError(
        "No MultiMend checkpoints found.\n"
        "Expected checkpoint directories containing trainer_state.json "
        "under /kaggle/input/datasets."
    )


print(f"\nFound {len(checkpoint_dirs)} checkpoint(s):")

for checkpoint in checkpoint_dirs:
    print(" -", checkpoint)


# ================================================================
# 4. Copy checkpoints into the repository
# ================================================================

print("\nPreparing model checkpoints...")

for checkpoint in checkpoint_dirs:

    destination = MODEL_DIR / checkpoint.name

    if destination.exists():
        print(f"Already exists: {destination.name}")
        continue

    print(f"Copying: {checkpoint.name}")

    shutil.copytree(
        checkpoint,
        destination
    )


# ================================================================
# 5. Verify final model directory
# ================================================================

installed_checkpoints = sorted(
    p for p in MODEL_DIR.glob("checkpoint-*")
    if p.is_dir() and (p / "trainer_state.json").is_file()
)

if not installed_checkpoints:
    raise FileNotFoundError(
        f"No valid checkpoints were installed in {MODEL_DIR}"
    )


print("\n" + "=" * 60)
print("Cell 3 completed successfully")
print("=" * 60)

print("\nQuixBugs:")
print("  Location:", QB_DIR)
print("  Buggy programs:", len(buggy_programs))
print("  Correct programs:", len(correct_programs))

print("\nModel:")
print("  Location:", MODEL_DIR)
print("  Checkpoints:", len(installed_checkpoints))

for checkpoint in installed_checkpoints:
    print("   -", checkpoint.name)


Repository: /kaggle/working/MultiMend28

QuixBugs benchmark: /kaggle/working/MultiMend28/benchmarks/QuixBugs
Buggy programs: 50
Correct programs: 50

Searching /kaggle/input/datasets for checkpoints...

Found 10 checkpoint(s):
 - /kaggle/input/datasets/honglcl/checkpoint-116200/checkpoint-116200
 - /kaggle/input/datasets/honglcl/checkpoint174300/checkpoint-174300
 - /kaggle/input/datasets/minhhanh123/checkpoint-232400/checkpoint-232400
 - /kaggle/input/datasets/minhhanh123/checkpoint290500/checkpoint-290500
 - /kaggle/input/datasets/thothanh28/checkpoint58100/checkpoint-58100
 - /kaggle/input/datasets/tramtvt/checkpoint348600/checkpoint-348600
 - /kaggle/input/datasets/tramtvt/checkpoint406700/checkpoint-406700
 - /kaggle/input/datasets/tramtvt/checkpoint464800/checkpoint-464800
 - /kaggle/input/datasets/tramtvt/checkpoint522900/checkpoint-522900
 - /kaggle/input/datasets/tramtvt/checkpoint581008/checkpoint-581008

Preparing model checkpoints...
Copying: checkpoint-116200
Copying: chec

In [12]:
# Cell 4 — Build QuixBugs-Python metadata WITHOUT RAG / Embedding

from pathlib import Path
import json
import difflib

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

quixbugs_dir = REPO / "benchmarks" / "QuixBugs"

# QuixBugs-Python thường có các thư mục này
buggy_dir = quixbugs_dir / "python_programs"
correct_dir = quixbugs_dir / "correct_python_programs"

output_dir = REPO / "outputs" / "QuixBugs-Python"
output_dir.mkdir(parents=True, exist_ok=True)

metadata_file = output_dir / "QuixBugs_Python.jsonl"
rem_file = output_dir / "rem.txt"
add_file = output_dir / "add.txt"

print("QuixBugs:", quixbugs_dir)
print("Buggy:", buggy_dir)
print("Correct:", correct_dir)


# ------------------------------------------------------------------
# Validate benchmark
# ------------------------------------------------------------------

if not buggy_dir.exists():
    raise FileNotFoundError(
        f"Buggy programs directory not found: {buggy_dir}"
    )

if not correct_dir.exists():
    raise FileNotFoundError(
        f"Correct programs directory not found: {correct_dir}"
    )


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def read_program(path):
    return path.read_text(encoding="utf-8").splitlines()


def unified_diff(old_lines, new_lines):
    """
    Return removed and added lines between buggy and correct versions.
    """
    diff = difflib.unified_diff(
        old_lines,
        new_lines,
        lineterm="",
    )

    removed = []
    added = []

    for line in diff:
        if line.startswith("---") or line.startswith("+++"):
            continue

        if line.startswith("-"):
            removed.append(line[1:])

        elif line.startswith("+"):
            added.append(line[1:])

    return removed, added


# ------------------------------------------------------------------
# Build metadata
# ------------------------------------------------------------------

records = []
all_removed = []
all_added = []

buggy_files = sorted(buggy_dir.glob("*.py"))

print("Buggy programs:", len(buggy_files))

for buggy_path in buggy_files:

    bugid = buggy_path.stem

    correct_path = correct_dir / buggy_path.name

    if not correct_path.exists():
        print(f"WARNING: missing correct program: {bugid}")
        continue

    buggy_lines = read_program(buggy_path)
    correct_lines = read_program(correct_path)

    removed, added = unified_diff(
        buggy_lines,
        correct_lines,
    )

    # Store global patch information
    all_removed.extend(removed)
    all_added.extend(added)

    record = {
        "bugid": bugid,
        "buggy_file": str(buggy_path),
        "correct_file": str(correct_path),
        "removed": removed,
        "added": added,
    }

    records.append(record)


# ------------------------------------------------------------------
# Write JSONL
# ------------------------------------------------------------------

with metadata_file.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


# ------------------------------------------------------------------
# Write rem.txt / add.txt
# ------------------------------------------------------------------

rem_file.write_text(
    "\n".join(all_removed) + ("\n" if all_removed else ""),
    encoding="utf-8",
)

add_file.write_text(
    "\n".join(all_added) + ("\n" if all_added else ""),
    encoding="utf-8",
)


# ------------------------------------------------------------------
# Validate
# ------------------------------------------------------------------

required_metadata = [
    metadata_file,
    rem_file,
    add_file,
]

missing = [
    str(path)
    for path in required_metadata
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        f"Metadata generation failed: {missing}"
    )

bug_count = sum(
    1
    for _ in metadata_file.open(encoding="utf-8")
)

print()
print("========================================")
print("Metadata generation complete")
print("========================================")
print("Metadata:", metadata_file)
print("rem.txt:", rem_file)
print("add.txt:", add_file)
print("Bug records:", bug_count)
print("Removed lines:", len(all_removed))
print("Added lines:", len(all_added))


QuixBugs: /kaggle/working/MultiMend28/benchmarks/QuixBugs
Buggy: /kaggle/working/MultiMend28/benchmarks/QuixBugs/python_programs
Correct: /kaggle/working/MultiMend28/benchmarks/QuixBugs/correct_python_programs
Buggy programs: 50

Metadata generation complete
Metadata: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/QuixBugs_Python.jsonl
rem.txt: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/rem.txt
add.txt: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/add.txt
Bug records: 50
Removed lines: 742
Added lines: 540


In [14]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)

result = subprocess.run(
    [sys.executable, "-m", "src.bugline_finders.quixbugs_python"],
    cwd=REPO,
    env=env,
    text=True,
)

print("Return code:", result.returncode)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Embedding...
bitcount 0
breadth_first_search 0
bucketsort 0
depth_first_search 0
detect_cycle 0
find_first_in_sorted 0
find_in_sorted 0
flatten 0
gcd 0
get_factors 0
hanoi 0
is_valid_parenthesization 0
kheapsort 0
knapsack 0
kth 0
lcs_length 0
levenshtein 0
lis 0
longest_common_subsequence 0
max_sublist_sum 0
mergesort 0
minimum_spanning_tree 0
next_palindrome 0
next_permutation 0
pascal 0
possible_change 0
powerset 0
quicksort 0
reverse_linked_list 0
rpn_eval 0
shortest_path_length 0
shortest_path_lengths 0
shortest_paths 0
shunting_yard 0
sieve 0
sqrt 0
subsequences 0
to_base 0
topological_ordering 0
wrap 0
Return code: 0


In [15]:
from src.configs import chromadb_path
import chromadb

print("ChromaDB path:", chromadb_path)

client = chromadb.PersistentClient(path=str(chromadb_path))

print("Collections hiện có:")
for collection in client.list_collections():
    print("-", collection.name)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


ChromaDB path: /kaggle/working/MultiMend28/chroma_db
Collections hiện có:
- QuixBugs-Python


In [16]:
# Cell 5 - Generate both experimental conditions from trainer-state checkpoints
import os
import subprocess
import sys

output_root = REPO / "outputs" / "QuixBugs-Python"


def run_generation(strategy: str) -> None:
    strategy_dir = output_root / f"outputs-multimend-{strategy}"
    for stale_file in (strategy_dir / "sequences_100.jsonl", strategy_dir / "generated_input.jsonl"):
        if stale_file.exists():
            stale_file.unlink()
    env = os.environ.copy()
    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy
    env["PYTHONPATH"] = str(REPO)
    subprocess.run([sys.executable, "-m", "src.generate_primary_candidates"], cwd=REPO, env=env, check=True)
    raw_file = strategy_dir / "sequences_100.jsonl"
    if not raw_file.is_file() or raw_file.stat().st_size == 0:
        raise RuntimeError(f"{strategy}: missing {raw_file}")
    print(f"{strategy}: {raw_file.stat().st_size:,} bytes")


for strategy in ("no_rag", "fixed_rag"):
    run_generation(strategy)


Starting no_rag...
Selected checkpoints:
- checkpoint-348600: global_step=348600
- checkpoint-406700: global_step=406700
- checkpoint-464800: global_step=464800
- checkpoint-522900: global_step=522900
- checkpoint-581008: global_step=581008
# RAG retrievals...
bitcount 0
breadth_first_search 0
bucketsort 0
depth_first_search 0
detect_cycle 0
find_first_in_sorted 0
find_in_sorted 0
flatten 0
gcd 0
get_factors 0
hanoi 0
is_valid_parenthesization 0
kheapsort 0
knapsack 0
kth 0
lcs_length 0
levenshtein 0
lis 0
longest_common_subsequence 0
max_sublist_sum 0
mergesort 0
minimum_spanning_tree 0
next_palindrome 0
next_permutation 0
pascal 0
possible_change 0
powerset 0
quicksort 0
reverse_linked_list 0
rpn_eval 0
shortest_path_length 0
shortest_path_lengths 0
shortest_paths 0
shunting_yard 0
sieve 0
sqrt 0
subsequences 0
to_base 0
topological_ordering 0
wrap 0
Tokenizing...


Map: 100%|██████████| 40/40 [00:00<00:00, 1042.65 examples/s]


Generating from checkpoint-348600...


Map: 100%|██████████| 40/40 [00:21<00:00,  1.84 examples/s]


Generating from checkpoint-406700...


...

Generating from checkpoint-522900...


Map: 100%|██████████| 40/40 [00:23<00:00,  1.70 examples/s]


Generating from checkpoint-581008...


Creating json from Arrow format: 100%|██████████| 20/20 [00:00<00:00, 386.66ba/s]


fixed_rag: generation complete (3,117,214 bytes)


In [17]:
# Cell 6 — Combine and deduplicate candidates for B0/B1

import os
import subprocess
import sys

STRATEGIES = ("no_rag", "fixed_rag")


def run_combine(strategy):
    output_dir = (
        REPO
        / "outputs"
        / "QuixBugs-Python"
        / f"outputs-multimend-{strategy}"
    )
    final_file = output_dir / "final_candidates_100.jsonl"

    output_dir.mkdir(parents=True, exist_ok=True)

    # Remove old combined output so we always combine
    # from the current generation results.
    if final_file.exists():
        final_file.unlink()
        print(f"{strategy}: removed old {final_file.name}")

    env = os.environ.copy()
    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy

    print(f"\nStarting combine for {strategy}...")

    subprocess.run(
        [sys.executable, "-m", "src.combine_checkpoints_results"],
        cwd=REPO,
        env=env,
        check=True,
    )

    if not final_file.is_file() or final_file.stat().st_size == 0:
        raise RuntimeError(
            f"{strategy}: combine did not create a valid output: {final_file}"
        )

    print(
        f"{strategy}: combine complete "
        f"({final_file.stat().st_size:,} bytes)"
    )


for strategy in STRATEGIES:
    run_combine(strategy)



Starting combine for no_rag...
All: 20000
Deduped: 9709
no_rag: combine complete (4,286,423 bytes)

Starting combine for fixed_rag...
All: 20000
Deduped: 9667
fixed_rag: combine complete (4,270,513 bytes)


In [40]:
import requests
from pathlib import Path

base_url = "https://raw.githubusercontent.com/ThaoTo2806/MultiMend28/main/"

files = [
    "src/validators/validate_runbugrunjs.py",
    "src/validators/validate_quixbugs_python.py"
]

project = Path("/kaggle/working/MultiMend28")

for file in files:
    url = base_url + file
    path = project / file

    response = requests.get(url)
    response.raise_for_status()

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(response.text, encoding="utf-8")

    print(f"✓ Loaded: {file}")

print("✓ Hoàn tất load từ GitHub")


✓ Loaded: src/validators/validate_runbugrunjs.py
✓ Loaded: src/validators/validate_quixbugs_python.py
✓ Hoàn tất load từ GitHub


In [70]:
# Cell 7A — Clean stale validator state

import shutil
from pathlib import Path

QUIXBUGS_OUTPUT = REPO / "outputs" / "QuixBugs-Python"

for strategy in ("no_rag", "fixed_rag"):
    output_dir = (
        QUIXBUGS_OUTPUT
        / f"outputs-multimend-{strategy}"
    )

    save_state_dir = output_dir / "save-state"
    result_file = output_dir / "plausible_candidates_100.jsonl"

    if save_state_dir.exists():
        shutil.rmtree(save_state_dir)
        print(f"Removed save-state: {save_state_dir}")

    # Xóa kết quả cũ để validator chạy lại
    if result_file.exists():
        result_file.unlink()
        print(f"Removed result: {result_file}")

print("Old validator state cleaned.")


Removed save-state: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend-no_rag/save-state
Removed result: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend-no_rag/plausible_candidates_100.jsonl
Removed save-state: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend-fixed_rag/save-state
Removed result: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend-fixed_rag/plausible_candidates_100.jsonl
Old validator state cleaned.


In [42]:
from pathlib import Path

QUIXBUGS_OUTPUT = Path("/kaggle/working/MultiMend28/outputs/QuixBugs-Python")

METADATA_FILE = QUIXBUGS_OUTPUT / "QuixBugs_Python.jsonl"
BACKUP_FILE = QUIXBUGS_OUTPUT / "QuixBugs_Python.jsonl.backup"

print("METADATA_FILE:")
print(METADATA_FILE)
print("exists:", METADATA_FILE.exists())
print("size:", METADATA_FILE.stat().st_size if METADATA_FILE.exists() else "N/A")

print("\nBACKUP_FILE:")
print(BACKUP_FILE)
print("exists:", BACKUP_FILE.exists())
print("size:", BACKUP_FILE.stat().st_size if BACKUP_FILE.exists() else "N/A")


METADATA_FILE:
/kaggle/working/MultiMend28/outputs/QuixBugs-Python/QuixBugs_Python.jsonl
exists: True
size: 40806

BACKUP_FILE:
/kaggle/working/MultiMend28/outputs/QuixBugs-Python/QuixBugs_Python.jsonl.backup
exists: True
size: 40806


In [68]:
from pathlib import Path

VALIDATOR = Path(
    "/kaggle/working/MultiMend28/src/validators/validate_quixbugs_python.py"
)

text = VALIDATOR.read_text(encoding="utf-8")

print("skipped_programs" in text)
print("detect_cycle_test" in text)
print("shortest_path_length_test" in text)

print("\n--- VALIDATOR CONFIG SECTION ---")

lines = text.splitlines()

for i, line in enumerate(lines):
    if "metadata_programs = len(bugs_metadata)" in line:
        print("\n".join(lines[i-3:i+7]))
        break


False
False
False

--- VALIDATOR CONFIG SECTION ---
    # Validate ALL programs.
    # No programs are skipped.

    metadata_programs = len(bugs_metadata)

    print(f"Metadata programs: {metadata_programs}")
    print(f"Programs to validate: {len(bugs_metadata)}")

    candidate_patches_df = pd.read_json(
        output_dir / f"final_candidates_{output_size}.jsonl",


In [47]:
from pathlib import Path

BASE = Path("/kaggle/working/MultiMend28/outputs/QuixBugs-Python")

print("Các thư mục/file trong QuixBugs-Python:\n")

for p in sorted(BASE.iterdir()):
    if p.is_dir():
        print("[DIR ]", p.name)
    else:
        print("[FILE]", p.name)


Các thư mục/file trong QuixBugs-Python:

[FILE] QuixBugs_Python.jsonl
[FILE] QuixBugs_Python.jsonl.backup
[FILE] add.txt
[FILE] context.txt
[FILE] context_log_fixed_rag.jsonl
[FILE] context_log_no_rag.jsonl
[DIR ] outputs-multimend-fixed_rag
[DIR ] outputs-multimend-no_rag
[FILE] rem.txt


In [ ]:
import contextlib
import json
import os
import shutil
import subprocess
import threading
import time
import timeit
from collections import ChainMap
from copy import deepcopy
from enum import Enum, auto
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

from ..configs import quixbugs_dir, quixbugs_genpy_dir


# ============================================================
# CONFIG
# ============================================================

project_dir = quixbugs_dir
gen_dir = quixbugs_genpy_dir

bugs_metadata_file = "QuixBugs_Python.jsonl"

model = "multimend"

context_strategy = os.environ.get(
    "MULTIMEND_CONTEXT_STRATEGY",
    "fixed_rag",
)

if context_strategy not in {
    "no_rag",
    "fixed_rag",
}:
    raise ValueError(
        f"Unsupported context strategy: {context_strategy}"
    )

output_dir = (
    gen_dir
    / f"outputs-{model}-{context_strategy}"
)

temp_dir = output_dir / "temp"
save_state_dir = output_dir / "save-state"

output_size = 100

TEST_TIMEOUT = float(
    os.environ.get(
        "MULTIMEND_VALIDATION_TIMEOUT",
        "60",
    )
)

N_WORKERS = int(
    os.environ.get(
        "MULTIMEND_VALIDATION_JOBS",
        "6",
    )
)

VALIDATION_SLEEP = float(
    os.environ.get(
        "MULTIMEND_VALIDATION_SLEEP",
        "1.0",
    )
)

RESET_VALIDATION = (
    os.environ.get(
        "MULTIMEND_RESET_VALIDATION",
        "0",
    ).lower()
    in {"1", "true", "yes"}
)


# ============================================================
# STATUS
# ============================================================

class Status(Enum):
    PLAUSIBLE = auto()
    PARSABLE = auto()
    TIMEOUT = auto()
    UNPARSABLE = auto()


# ============================================================
# TQDM / JOBLIB
# ============================================================

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    def tqdm_print_progress(self):
        if self.n_completed_tasks > tqdm_object.n:
            completed = (
                self.n_completed_tasks
                - tqdm_object.n
            )
            tqdm_object.update(
                n=completed
            )

    original_print_progress = (
        joblib.parallel.Parallel.print_progress
    )

    joblib.parallel.Parallel.print_progress = (
        tqdm_print_progress
    )

    try:
        yield tqdm_object
    finally:
        joblib.parallel.Parallel.print_progress = (
            original_print_progress
        )
        tqdm_object.close()


# ============================================================
# DATAFRAME HELPERS
# ============================================================

def get_hunk_candidates(
    df: pd.DataFrame,
    hunk: int,
) -> pd.DataFrame:
    return df.loc[
        df["hunk"] == hunk
    ]


def get_candidates(
    df: pd.DataFrame,
    bugid: str,
) -> pd.DataFrame:
    return df.loc[
        df["bugid"].astype(str)
        == str(bugid)
    ]


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_patch(value) -> str:
    """
    Normalize generated patch and ground-truth patch
    before exact-match comparison.
    """

    if value is None:
        return ""

    return (
        str(value)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
    )


# ============================================================
# GROUND TRUTH
# ============================================================

def load_ground_truth():
    """
    Build:

        (bugid, hunk) -> target patch

    from QuixBugs_Python.jsonl.
    """

    metadata_path = (
        gen_dir
        / bugs_metadata_file
    )

    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as meta_file:

        bugs_metadata = ChainMap(
            *[
                json.loads(line)
                for line in meta_file
            ][::-1]
        )

    ground_truth = {}

    for bugid, hunks in bugs_metadata.items():

        for hunk_id, hunk in enumerate(hunks):

            target = hunk.get(
                "added_lines",
                "",
            )

            if not target:
                target = hunk.get(
                    "target",
                    "",
                )

            ground_truth[
                (
                    str(bugid),
                    int(hunk_id),
                )
            ] = normalize_patch(
                target
            )

    return bugs_metadata, ground_truth


# ============================================================
# CORRECT
# ============================================================

def calculate_correct(
    df: pd.DataFrame,
    ground_truth: dict,
) -> pd.DataFrame:
    """
    Candidate is CORRECT iff its generated patch exactly
    matches the target patch in QuixBugs metadata.
    """

    def check_row(row):

        key = (
            str(row["bugid"]),
            int(row["hunk"]),
        )

        target = ground_truth.get(
            key,
            "",
        )

        generated = normalize_patch(
            row.get(
                "decoded_sequences",
                "",
            )
        )

        return (
            bool(target)
            and generated == target
        )

    df["correct"] = df.apply(
        check_row,
        axis=1,
    )

    return df


# ============================================================
# PATCH APPLICATION
# ============================================================

def insert_patch(
    patch,
    source_file_path,
    target_file_path,
    bug_line,
    bug_len,
    indent,
):
    """
    Apply one candidate to a clean copy of the buggy file.
    """

    with open(
        source_file_path,
        "r",
        encoding="utf-8",
    ) as file:
        lines = file.readlines()

    patch = normalize_patch(
        patch
    )

    replacement = (
        indent
        + patch
        + "\n"
    )

    if bug_len == 0:

        lines.insert(
            bug_line,
            replacement,
        )

    else:

        lines[
            bug_line - 1:
            (bug_line - 1) + bug_len
        ] = [
            replacement
        ]

    with open(
        target_file_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.writelines(lines)


# ============================================================
# TEST EXECUTION
# ============================================================

def run_tests(
    bugid: str,
    project_copy_dir: Path,
) -> Status:

    tests_dir = (
        project_copy_dir
        / "python_testcases"
    )

    test_file = (
        tests_dir
        / f"test_{bugid}.py"
    )

    args = [
        "pytest",
        "-x",
        str(test_file),
    ]

    try:

        result = subprocess.run(
            args,
            capture_output=True,
            timeout=TEST_TIMEOUT,
        )

    except subprocess.TimeoutExpired:

        return Status.TIMEOUT

    if result.returncode == 0:
        return Status.PLAUSIBLE

    if result.returncode == 2:
        return Status.UNPARSABLE

    return Status.PARSABLE


# ============================================================
# DATASET COPY
# ============================================================

def copy_dataset_files(
    dataset_dir,
    temp_dataset_dir,
):

    shutil.copytree(
        dataset_dir,
        temp_dataset_dir,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns(
            ".*"
        ),
    )


# ============================================================
# SAVE VALIDATION STATE
# ============================================================

def save_validation_state(
    df: pd.DataFrame,
    bugid: str,
):

    save_state_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    save_path = (
        save_state_dir
        / f"{bugid}.jsonl"
    )

    temp_path = (
        save_state_dir
        / f"{bugid}.jsonl.tmp"
    )

    df.to_json(
        temp_path,
        orient="records",
        lines=True,
    )

    temp_path.replace(
        save_path
    )


# ============================================================
# VALIDATE ONE BUG
# ============================================================

def apply_patch(
    cp_df: pd.DataFrame,
    bugid: str,
    hunks: list,
) -> Optional[pd.DataFrame]:

    save_file_path = (
        save_state_dir
        / f"{bugid}.jsonl"
    )

    if save_file_path.exists():
        return None

    # --------------------------------------------------------
    # QuixBugs original validator validates one hunk here.
    # --------------------------------------------------------

    if len(hunks) != 1:
        return None

    hunk = hunks[0]

    pid = threading.get_ident()

    project_copy_dir = (
        temp_dir
        / str(pid)
        / "QuixBugs"
    )

    copy_dataset_files(
        project_dir,
        project_copy_dir,
    )

    target_file_path = (
        project_copy_dir
        / "python_programs"
        / f"{bugid}.py"
    )

    source_file_path = (
        temp_dir
        / str(pid)
        / "sources"
        / bugid
        / f"{bugid}.py"
    )

    source_file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copyfile(
        target_file_path,
        source_file_path,
    )

    bug_line, bug_len = (
        hunk[
            "removed_line_numbers_range"
        ]
    )

    # --------------------------------------------------------
    # Preserve original indentation.
    # --------------------------------------------------------

    added_lines = hunk.get(
        "added_lines",
        "",
    )

    indent_size = (
        len(added_lines)
        - len(
            added_lines.lstrip(
                " \t"
            )
        )
    )

    indent = added_lines[
        :indent_size
    ]

    # --------------------------------------------------------
    # Validate candidates.
    # --------------------------------------------------------

    bug_hunk_subset_df = (
        get_hunk_candidates(
            cp_df,
            0,
        )
    )

    for index, patch in (
        bug_hunk_subset_df[
            "decoded_sequences"
        ].items()
    ):

        # IMPORTANT:
        # Always restore the original buggy program
        # before testing the next candidate.

        shutil.copyfile(
            source_file_path,
            target_file_path,
        )

        insert_patch(
            patch,
            source_file_path,
            target_file_path,
            bug_line,
            bug_len,
            indent,
        )

        start_timer = (
            timeit.default_timer()
        )

        status = run_tests(
            bugid,
            project_copy_dir,
        )

        end_timer = (
            timeit.default_timer()
        )

        cp_df.at[
            index,
            "validation_time",
        ] = (
            end_timer
            - start_timer
        )

        if status is Status.PLAUSIBLE:

            cp_df.at[
                index,
                "plausible",
            ] = True

            cp_df.at[
                index,
                "parsable",
            ] = True

            # Paper-style validation:
            # stop after first plausible candidate.

            break

        elif status is Status.PARSABLE:

            cp_df.at[
                index,
                "parsable",
            ] = True

        elif status is Status.TIMEOUT:

            cp_df.at[
                index,
                "timeout",
            ] = True

            cp_df.at[
                index,
                "parsable",
            ] = True

        if VALIDATION_SLEEP > 0:
            time.sleep(
                VALIDATION_SLEEP
            )

    save_validation_state(
        cp_df,
        bugid,
    )

    return cp_df


# ============================================================
# RESET
# ============================================================

def reset_validation_state():

    if RESET_VALIDATION:

        shutil.rmtree(
            save_state_dir,
            ignore_errors=True,
        )


# ============================================================
# MAIN
# ============================================================

def main():

    print(
        "\n"
        + "=" * 70
    )

    print(
        "MultiMend QuixBugs-Python Validator"
    )

    print(
        "=" * 70
    )

    print(
        f"context strategy : {context_strategy}"
    )

    print(
        f"output directory  : {output_dir}"
    )

    print(
        f"candidate size    : {output_size}"
    )

    print(
        f"timeout           : {TEST_TIMEOUT}s"
    )

    print(
        f"workers           : {N_WORKERS}"
    )

    # ========================================================
    # METADATA + GROUND TRUTH
    # ========================================================

    bugs_metadata, ground_truth = (
        load_ground_truth()
    )

    print(
        f"metadata bugs     : "
        f"{len(bugs_metadata)}"
    )

    print(
        f"ground-truth hunks: "
        f"{len(ground_truth)}"
    )

    # ========================================================
    # CANDIDATES
    # ========================================================

    candidate_file = (
        output_dir
        / f"final_candidates_{output_size}.jsonl"
    )

    if not candidate_file.exists():

        raise FileNotFoundError(
            "Candidate file not found:\n"
            f"{candidate_file}"
        )

    candidate_patches_df = (
        pd.read_json(
            candidate_file,
            orient="records",
            lines=True,
        )
    )

    print(
        f"candidate rows    : "
        f"{len(candidate_patches_df)}"
    )

    # ========================================================
    # VALIDATION COLUMNS
    # ========================================================

    candidate_patches_df[
        "plausible"
    ] = False

    candidate_patches_df[
        "parsable"
    ] = False

    candidate_patches_df[
        "timeout"
    ] = False

    candidate_patches_df[
        "validation_time"
    ] = np.nan

    # ========================================================
    # RESET
    # ========================================================

    reset_validation_state()

    shutil.rmtree(
        temp_dir,
        ignore_errors=True,
    )

    save_state_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ========================================================
    # VALIDATE
    # ========================================================

    with tqdm_joblib(
        tqdm(
            total=len(bugs_metadata),
            desc=f"Validating {context_strategy}",
        )
    ):

        Parallel(
            n_jobs=N_WORKERS,
            backend="threading",
        )(
            delayed(apply_patch)(
                deepcopy(
                    get_candidates(
                        candidate_patches_df,
                        bugid,
                    )
                ),
                bugid,
                hunks,
            )
            for bugid, hunks
            in bugs_metadata.items()
        )

    # ========================================================
    # LOAD STATES
    # ========================================================

    state_files = sorted(
        save_state_dir.glob(
            "*.jsonl"
        )
    )

    if not state_files:

        raise RuntimeError(
            "No validation states were produced."
        )

    cp_dfs = []

    for state_file in state_files:

        state_df = pd.read_json(
            state_file,
            orient="records",
            lines=True,
        )

        cp_dfs.append(
            state_df
        )

    concatenated_cp_df = pd.concat(
        cp_dfs,
        ignore_index=True,
    )

    if len(candidate_patches_df) != len(
        concatenated_cp_df
    ):

        raise RuntimeError(
            "Candidate count changed during validation: "
            f"{len(candidate_patches_df)} -> "
            f"{len(concatenated_cp_df)}"
        )

    # ========================================================
    # CALCULATE CORRECT
    # ========================================================

    concatenated_cp_df = calculate_correct(
        concatenated_cp_df,
        ground_truth,
    )

    # ========================================================
    # SAVE CANDIDATE RESULTS
    # ========================================================

    result_file = (
        output_dir
        / f"plausible_candidates_{output_size}.jsonl"
    )

    concatenated_cp_df.to_json(
        result_file,
        orient="records",
        lines=True,
    )

    # ========================================================
    # HUNK LEVEL
    # ========================================================

    correct_hunks = (
        concatenated_cp_df
        .groupby(
            [
                "bugid",
                "hunk",
            ]
        )["correct"]
        .any()
    )

    plausible_hunks = (
        concatenated_cp_df
        .groupby(
            [
                "bugid",
                "hunk",
            ]
        )["plausible"]
        .any()
    )

    total_hunks = len(
        plausible_hunks
    )

    correct_hunk_count = int(
        correct_hunks.sum()
    )

    plausible_hunk_count = int(
        plausible_hunks.sum()
    )

    # ========================================================
    # BUG LEVEL
    # ========================================================
    #
    # A bug is CORRECT when ALL of its hunks have
    # at least one correct candidate.
    #
    # A bug is PLAUSIBLE when ALL of its hunks have
    # at least one plausible candidate.
    # ========================================================

    correct_bugs = (
        correct_hunks
        .groupby("bugid")
        .all()
    )

    plausible_bugs = (
        plausible_hunks
        .groupby("bugid")
        .all()
    )

    total_bugs = len(
        plausible_bugs
    )

    correct_bug_count = int(
        correct_bugs.sum()
    )

    plausible_bug_count = int(
        plausible_bugs.sum()
    )

    # ========================================================
    # CANDIDATE LEVEL
    # ========================================================

    total_candidates = len(
        concatenated_cp_df
    )

    correct_candidate_count = int(
        concatenated_cp_df[
            "correct"
        ].sum()
    )

    plausible_candidate_count = int(
        concatenated_cp_df[
            "plausible"
        ].sum()
    )

    correct_and_plausible_count = int(
        (
            concatenated_cp_df["correct"]
            &
            concatenated_cp_df["plausible"]
        ).sum()
    )

    # ========================================================
    # PAPER METRICS
    # ========================================================

    paper_metrics = pd.DataFrame(
        [
            {
                "strategy": context_strategy,

                "total_bugs": total_bugs,
                "correct_bugs": correct_bug_count,
                "plausible_bugs": plausible_bug_count,

                "total_hunks": total_hunks,
                "correct_hunks": correct_hunk_count,
                "plausible_hunks": plausible_hunk_count,

                "total_candidates": total_candidates,
                "correct_candidates":
                    correct_candidate_count,
                "plausible_candidates":
                    plausible_candidate_count,
                "correct_and_plausible_candidates":
                    correct_and_plausible_count,

                "correct_bug_rate":
                    (
                        correct_bug_count
                        / total_bugs
                        if total_bugs
                        else 0.0
                    ),

                "plausible_bug_rate":
                    (
                        plausible_bug_count
                        / total_bugs
                        if total_bugs
                        else 0.0
                    ),

                "correct_hunk_rate":
                    (
                        correct_hunk_count
                        / total_hunks
                        if total_hunks
                        else 0.0
                    ),

                "plausible_hunk_rate":
                    (
                        plausible_hunk_count
                        / total_hunks
                        if total_hunks
                        else 0.0
                    ),
            }
        ]
    )

    paper_metrics_file = (
        output_dir
        / f"paper_bug_metrics_{output_size}.jsonl"
    )

    paper_metrics.to_json(
        paper_metrics_file,
        orient="records",
        lines=True,
    )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print()
    print(
        "=" * 70
    )

    print(
        f"RESULTS: {context_strategy}"
    )

    print(
        "=" * 70
    )

    print(
        f"BUGS       : {total_bugs}"
    )

    print(
        f"CORRECT    : "
        f"{correct_bug_count} / {total_bugs}"
        f" "
        f"({100 * correct_bug_count / total_bugs:.2f}%)"
        if total_bugs
        else "CORRECT    : 0 / 0"
    )

    print(
        f"PLAUSIBLE  : "
        f"{plausible_bug_count} / {total_bugs}"
        f" "
        f"({100 * plausible_bug_count / total_bugs:.2f}%)"
        if total_bugs
        else "PLAUSIBLE  : 0 / 0"
    )

    print()

    print(
        f"HUNKS      : {total_hunks}"
    )

    print(
        f"CORRECT    : "
        f"{correct_hunk_count} / {total_hunks}"
    )

    print(
        f"PLAUSIBLE  : "
        f"{plausible_hunk_count} / {total_hunks}"
    )

    print()

    print(
        f"CANDIDATES : {total_candidates}"
    )

    print(
        f"CORRECT    : "
        f"{correct_candidate_count}"
    )

    print(
        f"PLAUSIBLE  : "
        f"{plausible_candidate_count}"
    )

    print(
        f"CORRECT + PLAUSIBLE : "
        f"{correct_and_plausible_count}"
    )

    print()

    print(
        f"Candidates -> {result_file}"
    )

    print(
        f"Metrics    -> {paper_metrics_file}"
    )

    print(
        "=" * 70
    )


if __name__ == "__main__":
    main()

Metadata programs: 40
Programs to validate: 40
Skipped: []
Backed up metadata to: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/QuixBugs_Python.jsonl.backup

VALIDATING: no_rag
timeout per test : 10 seconds
sleep candidates : 0.05 seconds
workers          : 6
[VALIDATOR CONFIG] test_timeout=10.0s, candidate_sleep=0.05s, workers=6
Metadata programs: 40
Programs to validate: 40


  0%|          | 0/40 [00:00<?, ?it/s]

[VALIDATOR] bug=find_first_in_sorted candidates=240
[VALIDATOR] START index=1242
[VALIDATOR] bug=bucketsort candidates=241
[VALIDATOR] START index=464
[VALIDATOR] bug=depth_first_search candidates=282
[VALIDATOR] START index=705
[VALIDATOR] bug=breadth_first_search candidates=239
[VALIDATOR] START index=225
[VALIDATOR] bug=bitcount candidates=225
[VALIDATOR] START index=0
[VALIDATOR] bug=detect_cycle candidates=255
[VALIDATOR] START index=987
[VALIDATOR] START index=465
[VALIDATOR] START index=706
[VALIDATOR] START index=988
[VALIDATOR] START index=1243
[VALIDATOR] START index=226
[VALIDATOR] START index=466
[VALIDATOR] START index=989


  2%|▎         | 1/40 [00:07<04:33,  7.02s/it]

[VALIDATOR] bug=find_in_sorted candidates=237
[VALIDATOR] START index=1482
[VALIDATOR] START index=227
[VALIDATOR] START index=707
[VALIDATOR] START index=467


...

[VALIDATOR] START index=9036
[VALIDATOR] START index=9037
[VALIDATOR] START index=9038
[VALIDATOR] START index=9039
[VALIDATOR] START index=9040
[VALIDATOR] START index=9041
[VALIDATOR] START index=8503
[VALIDATOR] START index=9042
[VALIDATOR] START index=8504
[VALIDATOR] START index=9043
[VALIDATOR] START index=8505
[VALIDATOR] START index=9044
[VALIDATOR] START index=9045
[VALIDATOR] START index=9046
[VALIDATOR] START index=9047
[VALIDATOR] START index=9048
[VALIDATOR] START index=9049
[VALIDATOR] START index=9050
[VALIDATOR] START index=9051
[VALIDATOR] START index=8506
[VALIDATOR] START index=9052
[VALIDATOR] START index=9053
[VALIDATOR] START index=9054
[VALIDATOR] START index=9055
[VALIDATOR] START index=9056
[VALIDATOR] START index=9057
[VALIDATOR] START index=9058
[VALIDATOR] START index=9059
[VALIDATOR] START index=8507
[VALIDATOR] START index=9060
[VALIDATOR] START index=9061
[VALIDATOR] START index=9062
[VALIDATOR] START index=9063
[VALIDATOR] START index=9064
[VALIDATOR] ST

 98%|█████████▊| 39/40 [30:22<01:28, 88.60s/it]

[VALIDATOR] START index=8527
[VALIDATOR] START index=8528
[VALIDATOR] START index=8529
[VALIDATOR] START index=8530
[VALIDATOR] START index=8531
[VALIDATOR] START index=8532
[VALIDATOR] START index=8533
[VALIDATOR] START index=8534
[VALIDATOR] START index=8535
[VALIDATOR] START index=8536
[VALIDATOR] START index=8537
[VALIDATOR] START index=8538
[VALIDATOR] START index=8539
[VALIDATOR] START index=8540
[VALIDATOR] START index=8541
[VALIDATOR] START index=8542
[VALIDATOR] START index=8543
[VALIDATOR] START index=8544
[VALIDATOR] START index=8545
[VALIDATOR] START index=8546
[VALIDATOR] START index=8547
[VALIDATOR] START index=8548
[VALIDATOR] START index=8549
[VALIDATOR] START index=8550
[VALIDATOR] START index=8551
[VALIDATOR] START index=8552
[VALIDATOR] START index=8553
[VALIDATOR] START index=8554
[VALIDATOR] START index=8555
[VALIDATOR] START index=8556
[VALIDATOR] START index=8557
[VALIDATOR] START index=8558
[VALIDATOR] START index=8559
[VALIDATOR] START index=8560
[VALIDATOR] ST

100%|██████████| 40/40 [44:08<00:00, 66.22s/it] 



===== PLAUSIBLE =====
bugid
bitcount                       True
breadth_first_search           True
bucketsort                    False
depth_first_search             True
detect_cycle                   True
find_first_in_sorted           True
find_in_sorted                 True
flatten                        True
gcd                           False
get_factors                    True
hanoi                          True
is_valid_parenthesization      True
kheapsort                      True
knapsack                       True
kth                           False
lcs_length                     True
levenshtein                    True
lis                            True
longest_common_subsequence     True
max_sublist_sum               False
mergesort                      True
minimum_spanning_tree          True
next_palindrome                True
next_permutation               True
pascal                         True
possible_change                True
powerset                      False

In [74]:
# Cell 8 â€” Produce metrics required by the Stage 1 plan
import json
import pandas as pd

def load_jsonl(path):
    return pd.read_json(path, orient="records", lines=True)

def bug_success_at_k(frame, column, k):
    eligible = frame[frame["rank"] < k]
    per_hunk = eligible.groupby(["bugid", "hunk"])[column].any()
    return per_hunk.groupby(level=0).all()

def summarize(strategy):
    base = output_root / f"outputs-multimend-{strategy}"
    candidates = load_jsonl(base / "final_candidates_100.jsonl")
    validated = load_jsonl(base / "plausible_candidates_100.jsonl")
    if "rank" not in validated:
        raise ValueError(f"{strategy}: validator output has no rank column")
    if "exact_match" not in validated:
        validated["exact_match"] = False
    validated["exact_match"] = validated["exact_match"].fillna(False)
    if "correct" not in validated:
        validated["correct"] = False
    validated["correct"] = validated["correct"].fillna(False)
    validated["plausible"] = validated["plausible"].fillna(False)
    bugs = int(validated["bugid"].nunique())
    plausible_bugs = bug_success_at_k(validated, "plausible", 10**9)
    exact_bugs = bug_success_at_k(validated, "exact_match", 10**9)
    context_path = base / f"context_log_{strategy}.jsonl"
    context = load_jsonl(context_path) if context_path.exists() else pd.DataFrame()
    context_count = len(context)
    return {
        "strategy": strategy,
        "bugs": bugs,
        "hunks": int(validated[["bugid", "hunk"]].drop_duplicates().shape[0]),
       # "raw_candidates": int(load_jsonl(base / "sequences_100.jsonl").shape[0]),
        "final_candidates": int(candidates.shape[0]),
        "correct_at_1": int(bug_success_at_k(validated, "correct", 1).sum()),
        "correct_at_10": int(bug_success_at_k(validated, "correct", 10).sum()),
        "correct_at_100": int(bug_success_at_k(validated, "correct", 100).sum()),
        "plausible_at_1": int(bug_success_at_k(validated, "plausible", 1).sum()),
        "plausible_at_10": int(bug_success_at_k(validated, "plausible", 10).sum()),
        "plausible_at_100": int(bug_success_at_k(validated, "plausible", 100).sum()),
        "correct_bugs": int(bug_success_at_k(validated, "correct", 10**9).sum()),
        "plausible_bugs": int(plausible_bugs.sum()),
        "exact_match_bugs": int(exact_bugs.sum()),
        #"truncated_inputs": int(context["was_truncated"].sum()) if context_count else 0,
       # "truncate_percent": round(100 * float(context["was_truncated"].sum()) / context_count, 4) if context_count else 0.0,
        "retrieved_contexts": int((context["retrieved_count"] > 0).sum()) if context_count else 0,
        "local_context_at_risk": int(context["local_context_at_risk"].sum()) if context_count else 0,
       # "context_records": context_count,
    }

summary = pd.DataFrame([summarize("no_rag"), summarize("fixed_rag")])
for metric in ("correct_at_1", "correct_at_10", "correct_at_100", "correct_bugs", "plausible_bugs", "exact_match_bugs"):
    summary[f"{metric}_rate"] = summary[metric] / summary["bugs"]

def case_status(strategy):
    frame = load_jsonl(output_root / f"outputs-multimend-{strategy}" / "plausible_candidates_100.jsonl")
    frame["plausible"] = frame["plausible"].fillna(False)
    return frame.groupby(["bugid", "hunk"])["plausible"].any()

b0_status = case_status("no_rag")
b1_status = case_status("fixed_rag")
cases = pd.concat([b0_status.rename("b0_plausible"), b1_status.rename("b1_plausible")], axis=1).fillna(False)
cases["classification"] = "unchanged"
cases.loc[~cases.b0_plausible & cases.b1_plausible, "classification"] = "rag_helped"
cases.loc[cases.b0_plausible & ~cases.b1_plausible, "classification"] = "rag_harmed"
cases.reset_index().to_json(output_root / "stage1_rag_case_comparison.jsonl", orient="records", lines=True)
case_counts = cases["classification"].value_counts().to_dict()

summary_path = output_root / "stage1_b0_b1_comparison.json"
summary.to_json(summary_path, orient="records", indent=2)
(output_root / "stage1_rag_case_counts.json").write_text(json.dumps(case_counts, indent=2), encoding="utf-8")
display(summary)
print("RAG case counts:", case_counts)
print("Saved:", summary_path)


,strategy,bugs,hunks,final_candidates,correct_at_1,correct_at_10,correct_at_100,plausible_at_1,plausible_at_10,plausible_at_100,...,plausible_bugs,exact_match_bugs,retrieved_contexts,local_context_at_risk,correct_at_1_rate,correct_at_10_rate,correct_at_100_rate,correct_bugs_rate,plausible_bugs_rate,exact_match_bugs_rate
0,no_rag,40,40,9709,5,16,27,7,21,29,...,29,27,0,0,0.125,0.400,0.675,0.675,0.725,0.675
1,fixed_rag,40,40,9667,5,15,27,7,20,29,...,29,27,0,0,0.125,0.375,0.675,0.675,0.725,0.675


RAG case counts: {'unchanged': 40}
Saved: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/stage1_b0_b1_comparison.json


In [73]:
# Time-to-repair and GPU-plan report
import os
from pathlib import Path
import pandas as pd

def _time_to_first(frame, flag):
    work = frame.copy()
    if "validation_time" not in work:
        return pd.Series(dtype="float64")
    work["validation_time"] = pd.to_numeric(work["validation_time"], errors="coerce").fillna(0.0)
    work = work.sort_values(["bugid", "rank"])
    work["elapsed_seconds"] = work.groupby("bugid")["validation_time"].cumsum()
    return work[work[flag].fillna(False).astype(bool)].groupby("bugid")["elapsed_seconds"].min()

def _time_stats(values, name):
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return {f"{name}_seconds_{k}": None for k in ("min", "max", "median", "mean")}
    return {f"{name}_seconds_{k}": round(float(getattr(values, k)()), 4) for k in ("min", "max", "median", "mean")}

def _gpu_plan():
    path = Path(os.environ.get("MULTIMEND_PLAN_XLSX", "MultiMend_ke_hoach_2026-10-04_2026-10-11.xls.xlsx"))
    if not path.exists():
        raise FileNotFoundError(f"GPU plan workbook not found: {path}")
    import openpyxl
    book = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = list(book["2_Thuc_hien_giai_doan"].iter_rows(min_row=3, values_only=True))
    gpu_count = max(float(row[3]) for row in rows if row[3] is not None)
    return {"plan_file": str(path), "planned_sessions": len(rows), "planned_gpu_count": int(gpu_count), "planned_slot_hours": 12.0, "planned_gpu_hours": len(rows) * 12.0 * gpu_count}

reports = []
for strategy in ("no_rag", "fixed_rag"):
    base = output_root / f"outputs-multimend-{strategy}"
    frame = pd.read_json(base / "plausible_candidates_100.jsonl", orient="records", lines=True)
    frame["correct"] = frame["correct"].fillna(False).astype(bool)
    frame["plausible"] = frame["plausible"].fillna(False).astype(bool)
    row = {"strategy": strategy, "bugs": int(frame["bugid"].nunique()), "validation_records": int(len(frame))}
    row.update(_time_stats(_time_to_first(frame, "correct"), "time_to_correct"))
    row.update(_time_stats(_time_to_first(frame, "plausible"), "time_to_plausible"))
    row["validation_seconds_total"] = round(float(pd.to_numeric(frame["validation_time"], errors="coerce").sum()), 4)
    reports.append(row)
report = pd.DataFrame(reports)
for key, value in _gpu_plan().items():
    report[key] = value
report["gpu_hours_observed"] = pd.to_numeric(os.environ.get("MULTIMEND_GPU_HOURS"), errors="coerce")
report["gpu_hours_note"] = "planned capacity, not consumed usage"
report.to_json(output_root / "stage1_time_gpu_metrics.json", orient="records", indent=2)
display(report)


,strategy,bugs,validation_records,time_to_correct_seconds_min,time_to_correct_seconds_max,time_to_correct_seconds_median,time_to_correct_seconds_mean,time_to_plausible_seconds_min,time_to_plausible_seconds_max,time_to_plausible_seconds_median,time_to_plausible_seconds_mean,validation_seconds_total,plan_file,planned_sessions,planned_gpu_count,planned_slot_hours,planned_gpu_hours,gpu_hours_observed,gpu_hours_note
0,no_rag,40,9709,5.7027,741.8266,28.5952,80.3834,5.7027,741.8266,28.5952,76.7066,10443.8707,MultiMend_ke_hoach_2026-10-04_2026-10-11.xls.xlsx,16,30,12.0,5760.0,NaN,"planned capacity, not consumed usage"
1,fixed_rag,40,9667,5.4090,740.6907,35.5343,87.5021,5.4090,740.6907,35.5343,83.1393,10521.7609,MultiMend_ke_hoach_2026-10-04_2026-10-11.xls.xlsx,16,30,12.0,5760.0,NaN,"planned capacity, not consumed usage"


In [75]:
# Cell 9 — Final reproducibility checks and artifact manifest

import hashlib
import json
import subprocess
from datetime import datetime, timezone

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


run_started_at = datetime.now(timezone.utc).isoformat()
artifact_manifest = {}

SKIPPED_PROGRAMS = {
    "detect_cycle_test",
    "shortest_path_length_test",
}

for strategy in ("no_rag", "fixed_rag"):
    directory = output_root / f"outputs-multimend-{strategy}"

    required = [
        directory / "generated_input.jsonl",
        directory / "sequences_100.jsonl",
        directory / "final_candidates_100.jsonl",
        directory / "plausible_candidates_100.jsonl",
    ]

    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"{strategy} incomplete: {missing}"
        )

    # ---------------------------------------------------------
    # Load raw candidates.
    # load_jsonl() returns a pandas DataFrame.
    # ---------------------------------------------------------
    raw = load_jsonl(
        directory / "sequences_100.jsonl"
    )

    if raw.empty:
        raise RuntimeError(
            f"{strategy}: sequences_100.jsonl is empty"
        )

    # ---------------------------------------------------------
    # Check columns.
    # ---------------------------------------------------------
    required_raw_columns = {"bugid", "hunk"}
    missing_raw_columns = (
        required_raw_columns - set(raw.columns)
    )

    if missing_raw_columns:
        raise RuntimeError(
            f"{strategy}: raw output missing columns: "
            f"{sorted(missing_raw_columns)}"
        )

    # ---------------------------------------------------------
    # Count actual bugs/hunks.
    # Two programs are intentionally skipped.
    # ---------------------------------------------------------
    raw_bugids = set(raw["bugid"].dropna().unique())

    unexpected_skipped = (
        SKIPPED_PROGRAMS & raw_bugids
    )

    if unexpected_skipped:
        raise RuntimeError(
            f"{strategy}: skipped programs unexpectedly appear "
            f"in sequences_100.jsonl: "
            f"{sorted(unexpected_skipped)}"
        )

    hunk_keys = raw[
        ["bugid", "hunk"]
    ].drop_duplicates()

    expected_hunks = len(hunk_keys)
    expected_records = expected_hunks * 5 * 100

    print("=" * 70)
    print(strategy)
    print(f"bugs        : {len(raw_bugids)}")
    print(f"hunks       : {expected_hunks}")
    print(f"raw records : {len(raw)}")
    print(f"expected    : {expected_records}")
    print("=" * 70)

    if len(raw) != expected_records:
        raise RuntimeError(
            f"{strategy}: expected {expected_records} raw records "
            f"for {expected_hunks} hunks, found {len(raw)}"
        )

    # ---------------------------------------------------------
    # Verify validation output.
    # ---------------------------------------------------------
    validated = load_jsonl(
        directory / "plausible_candidates_100.jsonl"
    )

    if validated.empty:
        raise RuntimeError(
            f"{strategy}: plausible_candidates_100.jsonl is empty"
        )

    required_validation_columns = {
        "plausible",
        "correct",
    }

    missing_columns = (
        required_validation_columns
        - set(validated.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{strategy}: validation output missing columns: "
            f"{sorted(missing_columns)}"
        )

    artifact_manifest[strategy] = {
        "status": "complete",
        "bugs": len(raw_bugids),
        "hunks": expected_hunks,
        "raw_records": len(raw),
        "validation_records": len(validated),
        "skipped_programs": sorted(SKIPPED_PROGRAMS),
        "files": {
            path.name: sha256(path)
            for path in required
        },
    }


manifest = {
    "stage": 1,
    "dataset": "QuixBugs-Python",
    "strategies": [
        "no_rag",
        "fixed_rag",
    ],
    "seed": 42,
    "num_checkpoints": 5,
    "num_return_sequences": 100,
    "skipped_programs": sorted(SKIPPED_PROGRAMS),
    "rag": {
        "top_k": 5,
        "distance_threshold": 0.5,
    },
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO,
        text=True,
    ).strip(),
    "run_started_at": run_started_at,
    "run_finished_at": datetime.now(timezone.utc).isoformat(),
    "artifacts": artifact_manifest,
    "comparison_file": str(summary_path),
}


(output_root / "stage1_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2))
print("STAGE 1 — B0/B1 COMPLETE")


no_rag
bugs        : 40
hunks       : 40
raw records : 20000
expected    : 20000
fixed_rag
bugs        : 40
hunks       : 40
raw records : 20000
expected    : 20000
{
  "stage": 1,
  "dataset": "QuixBugs-Python",
  "strategies": [
    "no_rag",
    "fixed_rag"
  ],
  "seed": 42,
  "num_checkpoints": 5,
  "num_return_sequences": 100,
  "skipped_programs": [
    "detect_cycle_test",
    "shortest_path_length_test"
  ],
  "rag": {
    "top_k": 5,
    "distance_threshold": 0.5
  },
  "git_commit": "08c0c75f5331f54616a083267e501f21713c6047",
  "run_started_at": "2026-10-07T15:37:21.340203+00:00",
  "run_finished_at": "2026-10-07T15:37:21.781929+00:00",
  "artifacts": {
    "no_rag": {
      "status": "complete",
      "bugs": 40,
      "hunks": 40,
      "raw_records": 20000,
      "validation_records": 9709,
      "skipped_programs": [
        "detect_cycle_test",
        "shortest_path_length_test"
      ],
      "files": {
        "generated_input.jsonl": "1a61343616821b865f5c5d0c73bd359

Giai đoạn 0:

In [ ]:
# Cell 1 — Kaggle environment
import os, sys, subprocess, textwrap, json, pathlib, shutil, time
from pathlib import Path

print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import error:", repr(e))


In [ ]:
# Cell 2 — Clone ONLY the MultiMend repo first.
# We intentionally do not recurse through all benchmark submodules yet.

REPO = Path("/kaggle/working/MultiMend")
if REPO.exists():
    print("Repo already exists:", REPO)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/h4iku/MultiMend.git",
        str(REPO)
    ], check=True)

os.chdir(REPO)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Repo:", REPO)
print("Top-level:", [p.name for p in REPO.iterdir()])


In [ ]:
# Cell 3 — Inspect the official repo BEFORE running anything
from pathlib import Path

for p in sorted((REPO / "src").rglob("*.py")):
    print(p.relative_to(REPO))

print("\nrequirements.txt:")
print((REPO / "requirements.txt").read_text())


## Important implementation findings

The official generation script uses:

- `max_input_length = 512`
- `max_target_length = 256`
- `beam_size = 100`
- `num_return_sequences = 100`
- `batch_size = 1`
- `num_checkpoints = 5`

It loads the last five `checkpoint-*` directories under `models/multimend-codet5-small`, generates from every checkpoint, and writes `sequences_100.jsonl`.

The context augmentation is local RAG: `all-MiniLM-L6-v2` + persistent ChromaDB, not a paid LLM API.


In [ ]:
# Cell 4 — Install official MultiMend dependencies.
# Do NOT upgrade unrelated Kaggle packages.

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "-r",
        "requirements.txt",
        "more-itertools",
        "pygments",
    ],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("MultiMend dependency installation failed.")

print("MultiMend dependency installation complete.")

In [ ]:
# Cell 4b — Force the versions required by MultiMend
import subprocess
import sys

packages = [
    "numpy==1.26.4",
    "pandas==2.2.3",
    "torch==2.4.0",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        *packages,
    ],
    check=True,
)

print("Pinned NumPy/Pandas/Torch versions installed.")

In [ ]:
# Cell 4c — Match torchvision with MultiMend's torch 2.4.0

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "torchvision==0.19.0",
    ],
    check=True,
)

print("torchvision 0.19.0 installed.")

In [ ]:
# Fix NumPy for ChromaDB 0.5.5 / MultiMend
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
    ],
    check=True,
)

print("NumPy 1.26.4 installed.")


In [ ]:
import numpy as np

print("NumPy:", np.__version__)


In [2]:
import torch
import transformers
import datasets
import chromadb
import sentence_transformers
import tree_sitter
import unidiff
import more_itertools
import pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch: 2.4.0+cu121
transformers: 4.44.2
datasets: 2.21.0
chromadb: 0.5.5
sentence-transformers: 3.0.1
tree-sitter: unknown
CUDA: True
GPU: Tesla T4


In [3]:
# Cell 5 — Verify versions and direct imports
import torch, transformers, datasets, chromadb, sentence_transformers
import tree_sitter, unidiff, more_itertools, pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())


torch: 2.4.0+cu121
transformers: 4.44.2
datasets: 2.21.0
chromadb: 0.5.5
sentence-transformers: 3.0.1
tree-sitter: unknown
CUDA: True


In [ ]:
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend")

print("Repo exists:", REPO.exists())
print("Repo:", REPO)

if REPO.exists():
    print("Contents:")
    for p in list(REPO.iterdir())[:20]:
        print(" -", p.name)


In [ ]:
# Cell 6 — Download QuixBugs

from pathlib import Path
import subprocess
import shutil

REPO = Path("/kaggle/working/MultiMend")
QB = REPO / "benchmarks" / "QuixBugs"

if not REPO.exists():
    raise FileNotFoundError(
        f"MultiMend repo not found at {REPO}. "
        "Please run the clone cell first."
    )

if not (QB / "python_programs").exists():
    if QB.exists():
        shutil.rmtree(QB)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/jkoppel/QuixBugs.git",
        str(QB)
    ], check=True)

print("QuixBugs files:")
for d in ["python_programs", "correct_python_programs", "python_testcases"]:
    p = QB / d
    print(
        d,
        "=>", p.exists(),
        "count=", len(list(p.glob("*.py"))) if p.exists() else 0
    )


In [ ]:
# Check what MultiMend itself actually references
!grep -RniE "from_pretrained|snapshot_download|huggingface|checkpoint" \
    /kaggle/working/MultiMend/src \
    --include="*.py" | head -200


xác định chính xác workflow/config của tác giả

In [ ]:
# Cell 7a — Inspect MultiMend model/config paths

print("=== configs.py ===")
print((REPO / "src" / "configs.py").read_text())

print("\n=== generate_primary_candidates.py (relevant section) ===")
lines = (REPO / "src" / "generate_primary_candidates.py").read_text().splitlines()
for i, line in enumerate(lines):
    if 180 <= i <= 235:
        print(f"{i+1}: {line}")


In [ ]:
# Cell 7b — Inspect existing model/results artifacts

from pathlib import Path

print("=== models/ ===")
models_dir = REPO / "models"

if models_dir.exists():
    for p in models_dir.rglob("*"):
        if p.is_file():
            print(p.relative_to(REPO))
else:
    print("No models directory yet.")

print("\n=== results/ ===")
results_dir = REPO / "results"

for p in results_dir.rglob("*"):
    if p.is_file():
        print(p.relative_to(REPO))


In [ ]:
# # Cell 7 — Download the OFFICIAL MultiMend fine-tuned checkpoint repository.
# # No API key is required for a public Hugging Face repository.

# from huggingface_hub import snapshot_download

# MODEL_REPO = "h4iku/multimend-codet5-small"
# MODEL_DIR = REPO / "models" / "multimend-codet5-small"
# MODEL_DIR.mkdir(parents=True, exist_ok=True)

# snapshot_download(
#     repo_id=MODEL_REPO,
#     local_dir=str(MODEL_DIR),
#     local_dir_use_symlinks=False,
# )

# print("Model snapshot:", MODEL_DIR)
# print("Checkpoint directories:")
# for p in sorted(MODEL_DIR.glob("checkpoint-*")):
#     print(" ", p.name)


In [ ]:
# Cell 8 — Inspect the official MultiMend training configuration

train_file = REPO / "src" / "train_model.py"
lines = train_file.read_text().splitlines()

for i, line in enumerate(lines, start=1):
    print(f"{i:3}: {line}")


tính chính xác số sample + số training steps + save_steps từ chính 4 dataset mà code dùng. (Chạy CPU thôi, không tốn GPU)

In [ ]:
# Cell 9 — Inspect official CoCoNuT training dataset sizes

from datasets import load_dataset

dataset_names = {
    "Python": "h4iku/coconut_python2010_preprocessed",
    "Java": "h4iku/coconut_java2006_preprocessed",
    "JavaScript": "h4iku/coconut_javascript2010_preprocessed",
    "C": "h4iku/coconut_c2005_preprocessed",
}

for prefix, name in dataset_names.items():
    print(f"\nLoading {prefix}: {name}")
    ds = load_dataset(name, split="train")
    print("raw samples:", len(ds))
    print("columns:", ds.column_names)


đã biết raw dataset tổng cộng khoảng 2.64 triệu samples, nhưng code còn filter rất nhiều sample. Cần tính số sample sau filter, vì đó mới quyết định 30 GPU-hours có đủ không.

In [ ]:
# Cell 10 — Reproduce the official filtering step and calculate training steps

from math import ceil
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

model_checkpoint = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

dataset_names = {
    "Python": "h4iku/coconut_python2010_preprocessed",
    "Java": "h4iku/coconut_java2006_preprocessed",
    "JavaScript": "h4iku/coconut_javascript2010_preprocessed",
    "C": "h4iku/coconut_c2005_preprocessed",
}

max_input_length = 512
max_target_length = 256


def preprocess_function(prefix, examples):
    inputs = [
        f"{prefix} {ex.strip()} :".replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["rem"]
    ]

    targets = [
        ex.strip().replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["add"]
    ]

    contexts = [
        " ".join(ex.split()).replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["context"]
    ]

    inputs_contexts = [
        f"{src} {ctx}" for src, ctx in zip(inputs, contexts)
    ]

    model_inputs = tokenizer(
        inputs_contexts,
        max_length=max_input_length,
        truncation=True,
    )

    model_inputs["inputs_only"] = tokenizer(inputs).input_ids

    labels = tokenizer(targets)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


def allowed_length(example):
    return (
        len(example["inputs_only"]) <= max_input_length
        and 2 < len(example["labels"]) <= max_target_length
    )


filtered_datasets = []

for prefix, dataset_name in dataset_names.items():
    print(f"\nProcessing {prefix}...")

    raw_dataset = load_dataset(dataset_name, split="train")

    tokenized = raw_dataset.map(
        lambda ex: preprocess_function(prefix, ex),
        batched=True,
        remove_columns=raw_dataset.column_names,
        desc=f"Tokenizing {prefix}",
    )

    filtered = tokenized.filter(
        allowed_length,
        desc=f"Filtering {prefix}",
    ).remove_columns(["inputs_only"])

    print("Raw:", len(raw_dataset))
    print("After filter:", len(filtered))

    filtered_datasets.append(filtered)


concatenated_dataset = concatenate_datasets(filtered_datasets)

print("\n==============================")
print("FINAL DATASET")
print("==============================")
print("Total training samples:", len(concatenated_dataset))

train_batch_size = 8
epochs = 2

epoch_steps = ceil(len(concatenated_dataset) / train_batch_size)
train_steps = epoch_steps * epochs
save_steps = epoch_steps // 5

print("Steps / epoch:", epoch_steps)
print("Total training steps:", train_steps)
print("Checkpoint save_steps:", save_steps)
print("Expected checkpoints:", 10)


In [ ]:
# Cell 11 — Benchmark official MultiMend training speed
# Uses the exact official model/training batch size.
# Only 200 steps, then estimate full training time.

import time
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

model_checkpoint = "Salesforce/codet5-small"

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

benchmark_args = Seq2SeqTrainingArguments(
    output_dir=str(REPO / "benchmark_train"),
    per_device_train_batch_size=8,
    max_steps=200,
    fp16=True,
    learning_rate=1e-4,
    lr_scheduler_type="constant",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=benchmark_args,
    train_dataset=concatenated_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting 200-step benchmark...")
start = time.time()

trainer.train()

elapsed = time.time() - start

print("\n==============================")
print("BENCHMARK")
print("==============================")
print(f"200 steps: {elapsed / 60:.2f} minutes")
print(f"Average: {elapsed / 200:.3f} sec/step")

estimated_hours = (elapsed / 200) * 581008 / 3600

print(f"Estimated full 581,008 steps: {estimated_hours:.2f} GPU hours")
print(f"30-hour budget remaining margin: {30 - estimated_hours:.2f} hours")


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/thothanh28/"
    "checkpoint58100/checkpoint-58100"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-58100"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)

In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/honglcl/"
    "checkpoint-116200/checkpoint-116200"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-116200"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/honglcl/"
    "checkpoint174300/checkpoint-174300"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-174300"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/minhhanh123/"
    "checkpoint-232400/checkpoint-232400"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-232400"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)

In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/tramtvt/"
    "checkpoint348600/checkpoint-348600"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-348600"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/tramtvt/checkpoint406700/"
    "checkpoint-406700"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-406700"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if "406700" in d:
            print("FOUND:", os.path.join(root, d))


In [ ]:
import os

print("WORLD_SIZE:", os.environ.get("WORLD_SIZE"))
print("LOCAL_RANK:", os.environ.get("LOCAL_RANK"))
print("RANK:", os.environ.get("RANK"))


mục tiêu giữ nguyên training config của MultiMend và chỉ thêm cơ chế auto-resume, dùng cell này. Nó sẽ tìm checkpoint-* mới nhất trong models/multimend-codet5-small/; nếu chưa có thì train từ đầu, nếu có thì resume từ checkpoint đó.

In [ ]:
# Cell 12 — Official MultiMend training with automatic checkpoint resume
#
# IMPORTANT:
# - Keeps the official MultiMend training configuration unchanged.
# - Only adds automatic checkpoint detection/resume.
# - Run this cell again in a new Kaggle session after restoring the
#   checkpoint directory.
#
# Expected checkpoint layout:
# /kaggle/working/MultiMend/models/multimend-codet5-small/
# ├── checkpoint-58100/
# ├── checkpoint-116200/
# ├── ...
#
# Do NOT delete optimizer.pt, scheduler.pt, trainer_state.json, etc.
# from the checkpoint if you want a proper training resume.

from pathlib import Path
import re
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# ------------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------------

REPO = Path("/kaggle/working/MultiMend")
MODELS_ROOT = REPO / "models"
OUTPUT_DIR = MODELS_ROOT / "multimend-codet5-small"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO)
print("Training output:", OUTPUT_DIR)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


# ------------------------------------------------------------------
# 2. Auto-detect latest valid checkpoint
# ------------------------------------------------------------------

def get_latest_checkpoint(output_dir: Path):
    checkpoints = []

    for p in output_dir.glob("checkpoint-*"):
        if not p.is_dir():
            continue

        match = re.fullmatch(r"checkpoint-(\d+)", p.name)
        if match:
            step = int(match.group(1))
            checkpoints.append((step, p))

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])

    return checkpoints[-1][1]


latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if latest_checkpoint is None:
    print("\nNo existing checkpoint found.")
    print("Training will start from Salesforce/codet5-small.")
else:
    print("\nLatest checkpoint found:")
    print(" ", latest_checkpoint)

    # Useful sanity check
    required_files = [
        "trainer_state.json",
        "optimizer.pt",
        "scheduler.pt",
    ]

    missing = [
        f for f in required_files
        if not (latest_checkpoint / f).exists()
    ]

    if missing:
        raise RuntimeError(
            "Checkpoint appears incomplete.\n"
            f"Checkpoint: {latest_checkpoint}\n"
            f"Missing files: {missing}\n\n"
            "Do not resume from an incomplete checkpoint."
        )

    print("Checkpoint training state looks complete.")


# ------------------------------------------------------------------
# 3. Official base model
# ------------------------------------------------------------------

MODEL_CHECKPOINT = "Salesforce/codet5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)


# ------------------------------------------------------------------
# 4. IMPORTANT:
#    Reuse the tokenized/filtered training dataset from Cell 8/earlier.
#
#    This variable should already exist in the notebook:
#
#        concatenated_dataset
#
#    We deliberately do NOT recreate the 2.3M-example preprocessing here.
# ------------------------------------------------------------------

if "concatenated_dataset" not in globals():
    raise RuntimeError(
        "concatenated_dataset is not available.\n"
        "Run the official dataset preprocessing cell first."
    )

print("\nTraining dataset:")
print(concatenated_dataset)


# ------------------------------------------------------------------
# 5. Official MultiMend generation configuration
# ------------------------------------------------------------------

max_input_length = 512
max_target_length = 256

generation_config = GenerationConfig(
    max_length=max_target_length,
    min_length=0,
    early_stopping=True,
    num_beams=5,
)


# ------------------------------------------------------------------
# 6. Official MultiMend training configuration
# ------------------------------------------------------------------

train_batch_size = 8
eval_batch_size = 16

epochs = 2
lr = 1e-4

checkpoints_each_epoch = 5

epoch_steps = (
    len(concatenated_dataset) // train_batch_size
    + int(len(concatenated_dataset) % train_batch_size != 0)
)

train_steps = epoch_steps * epochs

save_steps = epoch_steps // checkpoints_each_epoch

print("\n==============================")
print("OFFICIAL TRAINING CONFIG")
print("==============================")
print("Dataset size:", len(concatenated_dataset))
print("Train batch size:", train_batch_size)
print("Epoch steps:", epoch_steps)
print("Total training steps:", train_steps)
print("Save steps:", save_steps)
print("Epochs:", epochs)
print("Learning rate:", lr)
print("FP16:", True)


# ------------------------------------------------------------------
# 7. Trainer
# ------------------------------------------------------------------

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    learning_rate=lr,

    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,

    save_total_limit=epochs * checkpoints_each_epoch,

    max_steps=train_steps,

    save_steps=save_steps,

    predict_with_generate=True,

    fp16=True,

    lr_scheduler_type="constant",

    generation_config=generation_config,

    report_to="tensorboard",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=concatenated_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


# ------------------------------------------------------------------
# 8. AUTO RESUME
# ------------------------------------------------------------------

print("\n==============================")
print("TRAINING")
print("==============================")

if latest_checkpoint is None:

    print("Starting training from scratch...")
    
    trainer.train(
        resume_from_checkpoint=False
    )

else:

    print(f"Resuming from: {latest_checkpoint}")

    trainer.train(
        resume_from_checkpoint=str(latest_checkpoint)
    )


# ------------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------------

print("\n==============================")
print("TRAINING SESSION FINISHED")
print("==============================")

latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if latest_checkpoint:
    print("Latest checkpoint:")
    print(latest_checkpoint)

    state_file = latest_checkpoint / "trainer_state.json"

    if state_file.exists():
        import json

        with open(state_file, "r") as f:
            state = json.load(f)

        print("Global step:", state.get("global_step"))
        print("Epoch:", state.get("epoch"))

print("\nAll checkpoint directories:")

for p in sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda x: int(x.name.split("-")[-1])
):
    print(" ", p.name)


In [ ]:
from pathlib import Path

ckpt = Path(
    "/kaggle/working/MultiMend/"
    "models/multimend-codet5-small/checkpoint-464800"
)

print("Exists:", ckpt.exists())

if ckpt.exists():
    print("Files:")
    for p in ckpt.iterdir():
        print(" -", p.name)


In [ ]:
import shutil
from pathlib import Path

ckpt = Path(
    "/kaggle/working/MultiMend/"
    "models/multimend-codet5-small/checkpoint-464800"
)

zip_path = Path("/kaggle/working/checkpoint-464800")

shutil.make_archive(
    str(zip_path),
    "zip",
    root_dir=ckpt.parent,
    base_dir=ckpt.name
)

final_zip = Path("/kaggle/working/checkpoint-464800.zip")

print("Created:", final_zip)
print("Size:", round(final_zip.stat().st_size / 1024**3, 2), "GB")


In [ ]:
!ls -lh /kaggle/working/kaggle_upload/


In [ ]:
# ==========================================
# CREATE KAGGLE DATASET - checkpoint464800
# ==========================================

import os
import json
import shutil

# ------------------------------
# 1. Cấu hình
# ------------------------------

USERNAME = "tramtvt"
DATASET_NAME = "checkpoint464800"

SOURCE_FILE = "/kaggle/working/checkpoint-464800.zip"
UPLOAD_DIR = "/kaggle/working/kaggle_upload"

DATASET_ID = f"{USERNAME}/{DATASET_NAME}"

# ------------------------------
# 2. Kiểm tra file checkpoint
# ------------------------------

if not os.path.isfile(SOURCE_FILE):
    raise FileNotFoundError(
        f"Không tìm thấy file:\n{SOURCE_FILE}"
    )

file_size = os.path.getsize(SOURCE_FILE)

print("Checkpoint found:")
print(SOURCE_FILE)
print(f"Size: {file_size / (1024**3):.3f} GB")

# ------------------------------
# 3. Xóa thư mục upload cũ
# ------------------------------

if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)

os.makedirs(UPLOAD_DIR, exist_ok=True)

# ------------------------------
# 4. Copy checkpoint
# ------------------------------

destination = os.path.join(
    UPLOAD_DIR,
    "checkpoint-464800.zip"
)

shutil.copy2(
    SOURCE_FILE,
    destination
)

# ------------------------------
# 5. Tạo metadata
# ------------------------------

metadata = {
    "title": DATASET_NAME,
    "id": DATASET_ID,
    "description": "Training checkpoint 464800",
    "licenses": [
        {
            "name": "Unknown"
        }
    ]
}

metadata_file = os.path.join(
    UPLOAD_DIR,
    "dataset-metadata.json"
)

with open(metadata_file, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

# ------------------------------
# 6. Kiểm tra trước khi upload
# ------------------------------

print("\n================================")
print("FILES TO UPLOAD")
print("================================")

for filename in os.listdir(UPLOAD_DIR):
    path = os.path.join(UPLOAD_DIR, filename)
    size = os.path.getsize(path)

    print(
        f"{filename:30s} "
        f"{size / (1024**2):.2f} MB"
    )

print("\n================================")
print("METADATA")
print("================================")

print(
    json.dumps(
        metadata,
        indent=2
    )
)


In [ ]:
!kaggle datasets create \
    -p /kaggle/working/kaggle_upload \
    --public


In [ ]:
import os
import shutil

# Xóa dataset-metadata.json
metadata = "/kaggle/working/dataset-metadata.json"
if os.path.exists(metadata):
    os.remove(metadata)

# Xóa thư mục kaggle_upload
upload_dir = "/kaggle/working/kaggle_upload"
if os.path.exists(upload_dir):
    shutil.rmtree(upload_dir)

print("Đã xóa.")


In [ ]:
!kaggle config view


In [ ]:
from pathlib import Path

zip_path = Path("/kaggle/working/checkpoint-174300.zip")

print("Exists:", zip_path.exists())

if zip_path.exists():
    print("Path:", zip_path)
    print("Size:", round(zip_path.stat().st_size / 1024**2, 2), "MB")
